## Table of Contents
This notebook requires both a Python kernel and an R kernel
* [Import Data](#fkt)
    * [Missings](#miss)
* [Sample description](#sample)
    * [Education and Age](#eduage)
    * [Parental disorder](#dis)
* [Multiple Regression Frequentists](#resfre)
    * [Simple Volume model](#volfreq)
        * [Assumption Check](#assumch)
    * [Simple Thickness model](#thickfreq)
        * [Assumption Check](#assumchth)
* [R - Hierarchical model Results](#res)
    * [Sample Comparsion](#samplecom)
    * [Hierarchical Volume model](#hiervol)
        * [Hierarchical Volume model Assumption](#assumvol)
    * [Hierarchical Thickness model](#hierthick)
        * [Hierarchical Thickness model Assumption](#assumthick)
* [Bayesian Hierarchical model Results](#bayesueber)
    * [Bayesian Hierarchical Volume model](#bayesvolu)
        * [Posterior Volume](#posteriorv)
        * [Assumption and Model Check](#assum)
     * [Bayesian Hierarchical Thickness model](#bayesthickness)
        * [Posterior Thickness](#posteriort)
        * [Assumption and Model Check](#assumT)





   
















### Import Data <a class="anchor" id="fkt"></a>

In [ ]:
#####################
####Python Kernel####
#####################

import os
import warnings
import pandas as pd
import numpy as np
import pingouin as pg
import seaborn as sns
import statsmodels
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import norm
from scipy.stats import zscore
warnings.filterwarnings('ignore') #deactivate warnings 

path='' # set path to data

stats_file = os.path.join(path)
df = pd.read_csv(stats_file,sep=',', decimal=".", encoding='utf-8')
df = df.replace(' ', np.nan)# set empty values to NaN, cause they missing
df['Kind1_CBCL.Tsum'] = pd.to_numeric(df['Kind1_CBCL.Tsum'],errors='coerce')#NaN ingore 

In [3]:
# base code
import numpy as np
import seaborn as sns
from statsmodels.tools.tools import maybe_unwrap_results
from statsmodels.graphics.gofplots import ProbPlot
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
from typing import Type


class LinearRegDiagnostic():
    """
    Diagnostic plots to identify potential problems in a linear regression fit.
    Mainly,
        a. non-linearity of data
        b. Correlation of error terms
        c. non-constant variance
        d. outliers
        e. high-leverage points
        f. collinearity

    Authors:
        Prajwal Kafle (p33ajkafle@gmail.com, where 3 = r)
        Does not come with any sort of warranty.
        Please test the code one your end before using.

        Matt Spinelli (m3spinelli@gmail.com, where 3 = r)
        (1) Fixed incorrect annotation of the top most extreme residuals in
            the Residuals vs Fitted and, especially, the Normal Q-Q plots.
        (2) Changed Residuals vs Leverage plot to match closer the y-axis
            range shown in the equivalent plot in the R package ggfortify.
        (3) Added horizontal line at y=0 in Residuals vs Leverage plot to
            match the plots in R package ggfortify and base R.
        (4) Added option for placing a vertical guideline on the Residuals
            vs Leverage plot using the rule of thumb of h = 2p/n to denote
            high leverage (high_leverage_threshold=True).
        (5) Added two more ways to compute the Cook's Distance (D) threshold:
            * 'baseR': D > 1 and D > 0.5 (default)
            * 'convention': D > 4/n
            * 'dof': D > 4 / (n - k - 1)
        (6) Fixed class name to conform to Pascal casing convention
        (7) Fixed Residuals vs Leverage legend to work with loc='best'
    """

    def __init__(self,
                 results: Type[statsmodels.regression.linear_model.RegressionResultsWrapper]) -> None:
        """
        For a linear regression model, generates following diagnostic plots:

        a. residual
        b. qq
        c. scale location and
        d. leverage

        and a table

        e. vif

        Args:
            results (Type[statsmodels.regression.linear_model.RegressionResultsWrapper]):
                must be instance of statsmodels.regression.linear_model object

        Raises:
            TypeError: if instance does not belong to above object

        Example:
        >>> import numpy as np
        >>> import pandas as pd
        >>> import statsmodels.formula.api as smf
        >>> x = np.linspace(-np.pi, np.pi, 100)
        >>> y = 3*x + 8 + np.random.normal(0,1, 100)
        >>> df = pd.DataFrame({'x':x, 'y':y})
        >>> res = smf.ols(formula= "y ~ x", data=df).fit()
        >>> cls = Linear_Reg_Diagnostic(res)
        >>> cls(plot_context="seaborn-v0_8")

        In case you do not need all plots you can also independently make an individual plot/table
        in following ways

        >>> cls = Linear_Reg_Diagnostic(res)
        >>> cls.residual_plot()
        >>> cls.qq_plot()
        >>> cls.scale_location_plot()
        >>> cls.leverage_plot()
        >>> cls.vif_table()
        """

        if isinstance(results, statsmodels.regression.linear_model.RegressionResultsWrapper) is False:
            raise TypeError("result must be instance of statsmodels.regression.linear_model.RegressionResultsWrapper object")

        self.results = maybe_unwrap_results(results)

        self.y_true = self.results.model.endog
        self.y_predict = self.results.fittedvalues
        self.xvar = self.results.model.exog
        self.xvar_names = self.results.model.exog_names

        self.residual = np.array(self.results.resid)
        influence = self.results.get_influence()
        self.residual_norm = influence.resid_studentized_internal
        self.leverage = influence.hat_matrix_diag
        self.cooks_distance = influence.cooks_distance[0]
        self.nparams = len(self.results.params)
        self.nresids = len(self.residual_norm)

    def __call__(self, plot_context='seaborn-v0_8', **kwargs):
        # print(plt.style.available)
        # GH#9157
        if plot_context not in plt.style.available:
            plot_context = 'default'
        with plt.style.context(plot_context):
            fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(10,10))
            self.residual_plot(ax=ax[0,0])
            self.qq_plot(ax=ax[0,1])
            self.scale_location_plot(ax=ax[1,0])
            self.leverage_plot(
                ax=ax[1,1],
                high_leverage_threshold = kwargs.get('high_leverage_threshold'),
                cooks_threshold = kwargs.get('cooks_threshold'))
            plt.show()

        return self.vif_table(), fig, ax,

    def residual_plot(self, ax=None):
        """
        Residual vs Fitted Plot

        Graphical tool to identify non-linearity.
        (Roughly) Horizontal red line is an indicator that the residual has a linear pattern
        """
        if ax is None:
            fig, ax = plt.subplots()

        sns.residplot(
            x=self.y_predict,
            y=self.residual,
            lowess=True,
            scatter_kws={'alpha': 0.5},
            line_kws={'color': 'red', 'lw': 1, 'alpha': 0.8},
            ax=ax)

        # annotations
        residual_abs = np.abs(self.residual)
        abs_resid = np.flip(np.argsort(residual_abs), 0)
        abs_resid_top_3 = abs_resid[:3]
        for i in abs_resid_top_3:
            ax.annotate(
                i,
                xy=(self.y_predict[i], self.residual[i]),
                color='C3')

        ax.set_title('Residuals vs Fitted', fontweight="bold")
        ax.set_xlabel('Fitted values')
        ax.set_ylabel('Residuals')
        return ax

    def qq_plot(self, ax=None):
        """
        Standarized Residual vs Theoretical Quantile plot

        Used to visually check if residuals are normally distributed.
        Points spread along the diagonal line will suggest so.
        """
        if ax is None:
            fig, ax = plt.subplots()

        QQ = ProbPlot(self.residual_norm)
        fig = QQ.qqplot(line='45', alpha=0.5, lw=1, ax=ax)

        # annotations
        abs_norm_resid = np.flip(np.argsort(np.abs(self.residual_norm)), 0)
        abs_norm_resid_top_3 = abs_norm_resid[:3]
        for i, x, y in self.__qq_top_resid(QQ.theoretical_quantiles, abs_norm_resid_top_3):
            ax.annotate(
                i,
                xy=(x, y),
                ha='right',
                color='C3')

        ax.set_title('Normal Q-Q', fontweight="bold")
        ax.set_xlabel('Theoretical Quantiles')
        ax.set_ylabel('Standardized Residuals')
        return ax

    def scale_location_plot(self, ax=None):
        """
        Sqrt(Standarized Residual) vs Fitted values plot

        Used to check homoscedasticity of the residuals.
        Horizontal line will suggest so.
        """
        if ax is None:
            fig, ax = plt.subplots()

        residual_norm_abs_sqrt = np.sqrt(np.abs(self.residual_norm))

        ax.scatter(self.y_predict, residual_norm_abs_sqrt, alpha=0.5);
        sns.regplot(
            x=self.y_predict,
            y=residual_norm_abs_sqrt,
            scatter=False, ci=False,
            lowess=True,
            line_kws={'color': 'red', 'lw': 1, 'alpha': 0.8},
            ax=ax)

        # annotations
        abs_sq_norm_resid = np.flip(np.argsort(residual_norm_abs_sqrt), 0)
        abs_sq_norm_resid_top_3 = abs_sq_norm_resid[:3]
        for i in abs_sq_norm_resid_top_3:
            ax.annotate(
                i,
                xy=(self.y_predict[i], residual_norm_abs_sqrt[i]),
                color='C3')

        ax.set_title('Scale-Location', fontweight="bold")
        ax.set_xlabel('Fitted values')
        ax.set_ylabel(r'$\sqrt{|\mathrm{Standardized\ Residuals}|}$');
        return ax

    def leverage_plot(self, ax=None, high_leverage_threshold=False, cooks_threshold='baseR'):
        """
        Residual vs Leverage plot

        Points falling outside Cook's distance curves are considered observation that can sway the fit
        aka are influential.
        Good to have none outside the curves.
        """
        if ax is None:
            fig, ax = plt.subplots()

        ax.scatter(
            self.leverage,
            self.residual_norm,
            alpha=0.5);

        sns.regplot(
            x=self.leverage,
            y=self.residual_norm,
            scatter=False,
            ci=False,
            lowess=True,
            line_kws={'color': 'red', 'lw': 1, 'alpha': 0.8},
            ax=ax)

        # annotations
        leverage_top_3 = np.flip(np.argsort(self.cooks_distance), 0)[:3]
        for i in leverage_top_3:
            ax.annotate(
                i,
                xy=(self.leverage[i], self.residual_norm[i]),
                color = 'C3')

        factors = []
        if cooks_threshold == 'baseR' or cooks_threshold is None:
            factors = [1, 0.5]
        elif cooks_threshold == 'convention':
            factors = [4/self.nresids]
        elif cooks_threshold == 'dof':
            factors = [4/ (self.nresids - self.nparams)]
        else:
            raise ValueError("threshold_method must be one of the following: 'convention', 'dof', or 'baseR' (default)")
        for i, factor in enumerate(factors):
            label = "Cook's distance" if i == 0 else None
            xtemp, ytemp = self.__cooks_dist_line(factor)
            ax.plot(xtemp, ytemp, label=label, lw=1.25, ls='--', color='red')
            ax.plot(xtemp, np.negative(ytemp), lw=1.25, ls='--', color='red')

        if high_leverage_threshold:
            high_leverage = 2 * self.nparams / self.nresids
            if max(self.leverage) > high_leverage:
                ax.axvline(high_leverage, label='High leverage', ls='-.', color='purple', lw=1)

        ax.axhline(0, ls='dotted', color='black', lw=1.25)
        ax.set_xlim(0, max(self.leverage)+0.01)
        ax.set_ylim(min(self.residual_norm)-0.1, max(self.residual_norm)+0.1)
        ax.set_title('Residuals vs Leverage', fontweight="bold")
        ax.set_xlabel('Leverage')
        ax.set_ylabel('Standardized Residuals')
        plt.legend(loc='best')
        return ax

    def vif_table(self):
        """
        VIF table

        VIF, the variance inflation factor, is a measure of multicollinearity.
        VIF > 5 for a variable indicates that it is highly collinear with the
        other input variables.
        """
        vif_df = pd.DataFrame()
        vif_df["Features"] = self.xvar_names
        vif_df["VIF Factor"] = [variance_inflation_factor(self.xvar, i) for i in range(self.xvar.shape[1])]

        return (vif_df
                .sort_values("VIF Factor")
                .round(2))


    def __cooks_dist_line(self, factor):
        """
        Helper function for plotting Cook's distance curves
        """
        p = self.nparams
        formula = lambda x: np.sqrt((factor * p * (1 - x)) / x)
        x = np.linspace(0.001, max(self.leverage), 50)
        y = formula(x)
        return x, y


    def __qq_top_resid(self, quantiles, top_residual_indices):
        """
        Helper generator function yielding the index and coordinates
        """
        offset = 0
        quant_index = 0
        previous_is_negative = None
        for resid_index in top_residual_indices:
            y = self.residual_norm[resid_index]
            is_negative = y < 0
            if previous_is_negative == None or previous_is_negative == is_negative:
                offset += 1
            else:
                quant_index -= offset
            x = quantiles[quant_index] if is_negative else np.flip(quantiles, 0)[quant_index]
            quant_index += 1
            previous_is_negative = is_negative
            yield resid_index, x, y



##### Missings <a class="anchor" id="miss"></a>

In [ ]:
def show_missing(df):
    """Return a Pandas dataframe describing the contents of a source dataframe including missing values."""
    pd.options.mode.use_inf_as_na = True   #to see inf values as NaN
    variables = []
    dtypes = []
    count = []
    missing = []
    pc_missing = []

    for item in df.columns:
        variables.append(item)
        dtypes.append(df[item].dtype)
        count.append(len(df[item]))
        missing.append(df[item].isna().sum())
        pc_missing.append(round((df[item].isna().sum() / len(df[item])) * 100, 2))

    output = pd.DataFrame({
    'variable': variables, 
    'dtype': dtypes,
    'count': count,
    'missing': missing, 
    'pc_missing': pc_missing
    })    

    print(output.loc[output['missing'] > 1])

show_missing(df)

### Sample description <a class="anchor" id="sample"></a>

In [ ]:
#Sample description, amount, age, gender, sibling, CBCL.Tsum
print('Anzahl EG: ' + str(df[df['Gruppenzugehörigkeit']=='EG'].shape[0]) + '\nMittleres Alter: ' + str(round(df['AlterJahreMonate'][df['Gruppenzugehörigkeit']=='EG'].mean(),3))+ ' (std: ' +str(round(df['AlterJahreMonate'][df['Gruppenzugehörigkeit']=='EG'].std(),3))+ ')'+ '\nGeschlecht: ' + str(df[df['Gruppenzugehörigkeit']=='EG'][df['Geschlecht']=='weiblich'].shape[0])+' weiblich und '+ str(df[df['Gruppenzugehörigkeit']=='EG'][df['Geschlecht']=='männlich'].shape[0]) + ' männlich'+ ' \nGeschwister: ' + str(df[df['Geschwisterkind']=='ja'][df['Gruppenzugehörigkeit']=='EG'].shape[0])+ ' (5 Geschwisterpaare) \nCBCL T-Wert Gesamtskala: ' + str(round(df['Kind1_CBCL.Tsum'][df['Gruppenzugehörigkeit']=='EG'].mean(skipna=True),3))+ ' (std: ' +str(round(df['Kind1_CBCL.Tsum'][df['Gruppenzugehörigkeit']=='EG'].std(),3))+')'+ '\nSES Gesamtskala: ' + str(round(df['SES_2'][df['Gruppenzugehörigkeit']=='EG'].mean(),3))+ ' (std: ' +str(round(df['SES_2'][df['Gruppenzugehörigkeit']=='EG'].std(),3))+ ')')      
print('\n\nAnzahl KG: ' + str(df[df['Gruppenzugehörigkeit']=='KG'].shape[0]) + '\nMittleres Alter: ' + str(round(df['AlterJahreMonate'][df['Gruppenzugehörigkeit']=='KG'].mean(),3)) + ' (std: ' +str(round(df['AlterJahreMonate'][df['Gruppenzugehörigkeit']=='KG'].std(),3)) + ')' + '\nGeschlecht: ' + str(df[df['Gruppenzugehörigkeit']=='KG'][df['Geschlecht']=='weiblich'].shape[0])+' weiblich und ' + str(df[df['Gruppenzugehörigkeit']=='KG'][df['Geschlecht']=='männlich'].shape[0]) + ' männlich' + '\nGeschwister: ' + str(df[df['Geschwisterkind']=='ja'][df['Gruppenzugehörigkeit']=='KG'].shape[0]) + ' (6 Geschwisterpaare 2 Dreigeschwister)' '\nCBCL T-Wert Gesamtskala: ' + str(round(df['Kind1_CBCL.Tsum'][df['Gruppenzugehörigkeit']=='KG'].mean(),3))+ ' (std: ' +str(round(df['Kind1_CBCL.Tsum'][df['Gruppenzugehörigkeit']=='KG'].std(),3))+ '\nSES Gesamtskala: ' + str(round(df['SES_2'][df['Gruppenzugehörigkeit']=='KG'].mean(),3))+ ' (std: ' +str(round(df['SES_2'][df['Gruppenzugehörigkeit']=='KG'].std(),3))+ ')\n\n')
aseg_all = df
print("Tage zwischen MRT und Labor: \nmedian: " +str(df['daysbetweenmriandlabor'].median()) + ' \nmin Tage: '+str(df['daysbetweenmriandlabor'].min()) + '\nmax Tage: '+str(df['daysbetweenmriandlabor'].max()))

##### Education and age <a class="anchor" id="eduage"></a>

In [ ]:
#pie chart function, % and total sum are shown
def make_autopct(values):
    def my_autopct(pct):
        total = sum(values)
        val = int(round(pct*total/100.0))
        return '{p:.2f}% ({v:d})'.format(p=pct,v=val)#{p:.2f}% if %is needed
    return my_autopct

#subplot 
fig1, (ax1, ax2) = plt.subplots(1,2, figsize=(18,10)) 
fig1.tight_layout(w_pad=12)

#plot histogram with fitted norm curve
mu, std = norm.fit(aseg_all['AlterJahreMonate'])
ax2.hist(aseg_all['AlterJahreMonate'], bins=15, alpha=0.6, edgecolor='black',density=True)
ax2.set_title('Age histogram of children')
ax2.set_xlabel('Age')
ax2.set_ylabel('density')
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, std)
ax2.plot(x, p, 'k', linewidth=2)

#plot pie chart
af = aseg_all.groupby(['Kind1_schule'])['Kind1_schule'].count().reset_index(name='count')
ax1.pie(af['count'], labels = af['Kind1_schule'],
       autopct=make_autopct(af['count']),
       wedgeprops = {"edgecolor" : "black", 'linewidth': 0.5, 'antialiased': True})
ax1.set_title("Education")
plt.rc ('font', size = 25) 

dfe = aseg_all[aseg_all['Gruppenzugehörigkeit'] == 'EG']#only EG Group
dfk = aseg_all[aseg_all['Gruppenzugehörigkeit'] == 'KG']#only KG Group

#count and %, children schoolform for EG and KG 
counts = dfe['Kind1_schule'].value_counts()
percs = dfe['Kind1_schule'].value_counts(normalize=True).mul(100).round(2)
print('experimental group\n' + str(pd.concat([counts,percs], axis=1, keys=['count', 'percentage'])))

counts = dfk['Kind1_schule'].value_counts()
percs = dfk['Kind1_schule'].value_counts(normalize=True).mul(100).round(2)
print('\n\ncontrol group\n' + str(pd.concat([counts,percs], axis=1, keys=['count', 'percentage'])))

##### Parental disorder <a class="anchor" id="dis"></a>

In [ ]:
dfe['Patient_Alter_or_moth'] = pd.to_numeric(dfe['Patient_Alter_or_moth'],errors='coerce')#NaN ingore 
dfk['Patient_Alter_or_moth'] = pd.to_numeric(dfk['Patient_Alter_or_moth'],errors='coerce')#NaN ingore 
print('ALter: ' + str(round(dfe['Patient_Alter_or_moth'][dfe['Gruppenzugehörigkeit']=='EG'].mean(),2)) + ' STD: '+  str(round(dfe['Patient_Alter_or_moth'][dfe['Gruppenzugehörigkeit']=='EG'].std(),2)))
print(str(dfe['Patient_geschlecht'][dfe['Gruppenzugehörigkeit']=='EG'].value_counts()))

print('\nMütter der KG Alter: ' + str(round(dfk['Patient_Alter_or_moth'][dfk['Gruppenzugehörigkeit']=='KG'].mean(),2)) + ' STD: '+  str(round(dfk['Patient_Alter_or_moth'][dfk['Gruppenzugehörigkeit']=='KG'].std(),2)))

#plot pie chart with parent illnes
af = df.groupby(['Patient_PrimDiagnoseT1.ICD10'])['Patient_PrimDiagnoseT1.ICD10'].count().reset_index(name='count')

fig, ax = plt.subplots(figsize =(24, 12))
fig.set_facecolor('white')
colors = ['red','green','green','green','orange','orange','purple','beige','beige','violet','violet','yellow','yellow','yellow','turquoise']
explode = (0.2, 0.05, 0.05, 0.05, 0, 0, 0.2, 0, 0, 0.1, 0.1, 0, 0, 0, 0.25,) 

wedges, texts, autotexts = ax.pie(af['count'], labels = af['Patient_PrimDiagnoseT1.ICD10'],
                                  colors =colors,
                                  autopct=make_autopct(af['count']),
                                  explode =explode,
                                  wedgeprops = {"edgecolor" : "black", 'linewidth': 0.5, 'antialiased': True})

ax.set_title("Parental disorders")
plt.rc ('font', size = 15) 
#ax.legend(wedges, af['Patient_PrimDiagnoseT1.ICD10'],
          #title ='Parental disorders',
          #loc ="center left",
          #bbox_to_anchor =(1.25, 0, 0.5, 1))

          
plt.show() 

print('Mood [affective] disorders & Neurotic, stress-related and somatoform disorders')
print('Delusional disorder vs. Depressive Episode vs. Recurrent depressive disorder vs. Dysthymia vs. Phobic anxiety disorders vs. Other anxiety disorders vs. Reaction to severe stress, and adjustment disorders vs Somatoform disorders')

#plot how many parental diagnosis exist and which one
#fig, axes = plt.subplots(5, 1, figsize=(18, 10))
#fig.set_figwidth(24)
#fig.set_figheight(18)
#fig.tight_layout(w_pad=12)

#sns.countplot(y=df['anzahl_diagT1'],ax=axes[0])
#sns.countplot(y=df['Patient_sek1DiagnoseT1'],alpha=1,ax=axes[1])
#sns.countplot(y=df['Patient_sek2DiagnoseT1'],alpha=1,ax=axes[2])
#sns.countplot(y=df['Patient_sek3DiagnoseT1'],alpha=1,ax=axes[3])
#sns.countplot(y=df['Patient_sek4DiagnoseT1'],alpha=1,ax=axes[4])



print('\n\nNumber of diagnoses:\n' + str(df['anzahl_diagT1'].value_counts()))
print('\n\nSecond diagnosis:\n' + str(df['Patient_sek1DiagnoseT1'].value_counts()))
print('\n\nThird diagnosis:\n' + str(df['Patient_sek2DiagnoseT1'].value_counts()))
print('\n\nFourth diagnosis:\n' + str(df['Patient_sek3DiagnoseT1'].value_counts()))
print('\n\nFifth diagnosis:\n' + str(df['Patient_sek4DiagnoseT1'].value_counts()))

print('\n\nmostly F3 and F4 disorders - depressive episode, Phobic anxiety disorders, anxiety disorders')


## Multiple Regression Frequentist <a class="anchor" id="resfre"></a>


### Simple Volumen model <a class="anchor" id="volfreq"></a>

In [ ]:
def format_ols_results(para):
    dff = pd.DataFrame() 
    dff['Param'] = res.params
    dff['tvalues'] = res.tvalues
    dff['pvalues'] = res.pvalues
    dff['rsquared'] = res.rsquared
    dff['rsquared_adj'] = res.rsquared_adj
    return dff

def plotRoi(roi_cols,predictor,Atlas,VolSuf,A,height): 
    n_comparisons =  len(roi_cols) 
    alpha = 0.05
    alpha_corr = 0.05 / n_comparisons # change Bonferroni
    g = sns.catplot(x='pvalues', y='Areal', kind='bar', height=height, data=A)
    g.set(xscale='log', xlim=(1e-5,2))
    sns.set(rc={'figure.figsize':(30,24)})
    print('Prädiktor '+ predictor+ ' auf ' + Atlas +'-Atlas '+ VolSuf + ' als Kirterium mit den Covariaten ' + covariates+ ' und p_value_corr: ' + str(alpha_corr))

    for ax in g.axes.flat:
        ax.axvline(alpha, ls='--',c='tomato')
        ax.axvline(alpha_corr, ls='--',c='darkred')
        

#z-transform the numeric data      
zdf=df
numeric_cols = zdf.select_dtypes(include=[np.number]).columns 
zdf[numeric_cols] = zdf[numeric_cols].apply(zscore)
zdf = zdf.replace(['EG','KG'],[1.0,0.0])


vektor = ("RightAmygdala","LeftAmygdala","RightHippocampus","LeftHippocampus","CCPosterior","CCMidPosterior","CCCentral","CCMidAnterior","CCAnterior","LinsulaVN","RinsulaVN")
predictor = 'Gruppenzugehörigkeit'
covariates = ('SES_2 + AlterJahreMonate  + eTIV  + C(Standort,Treatment(reference="Dortmund")) + C(Geschlecht,Treatment(reference="männlich")) ') #Probanden_ID + SES_2

sns.set_style("darkgrid")

ols_df = pd.DataFrame()
for response in vektor:
    res = smf.ols(f"{response} ~ {covariates} + {predictor}", data=zdf).fit()
    res_df = format_ols_results(res)
    res_df['Areal'] = response
    ols_df = pd.concat([res_df,ols_df], axis=0, join='outer')
A = ols_df.loc['Gruppenzugehörigkeit']
plotRoi(vektor,predictor,'ASEG_FREE','(Volume)',A,5) 
#multipletests(A['pvalues'],method='fdr_bh', is_sorted=False, returnsorted=False)        

#to have a look at the complete regression model
#response = "RightAmygdala"
#results = smf.ols(f"{response} ~ {covariates} + {predictor}", data=zdf).fit()
#results.summary()
#results.model.data.orig_exog[:] # Designmatrix

### Assumption Check <a class="anchor" id="assumch"></a>

In [ ]:
for response in vektor:
    res = smf.ols(f"{response} ~ {covariates} + {predictor}", data=zdf).fit()
    print('\n\n' +"\033[1m"+"\033[1;37;40m" ,response,"\033[0m")
    #print(res.summary())
    cls = LinearRegDiagnostic(res)
    vif, fig, ax = cls()
    print(vif)

## Simple Thickness model <a class="anchor" id="thickfreq"></a>

In [ ]:
def plotRoi(roi_cols,predictor,Atlas,VolSuf,A,height): 
    n_comparisons =  len(roi_cols) 
    alpha = 0.05
    alpha_corr = 0.05 / (n_comparisons*2) # change Bonferroni
    g = sns.catplot(x='pvalues', y='Areal', kind='bar', height=height, data=A)
    g.set(xscale='log', xlim=(1e-5,2))
    sns.set(rc={'figure.figsize':(30,24)})
    print('Prädiktor '+ predictor+ ' auf ' + Atlas +'-Atlas '+ VolSuf + ' als Kirterium mit den Covariaten ' + covariates+ ' und p_value_corr: ' + str(alpha_corr))

    for ax in g.axes.flat:
        ax.axvline(alpha, ls='--',c='tomato')
        ax.axvline(alpha_corr, ls='--',c='darkred')


#Temporal ROIS
Tvektor = ('LentorhinalT','LfusiformT', 'LinferiortemporalT','LtransversetemporalT', 'LtemporalpoleT','LsuperiortemporalT','LmiddletemporalT','LparahippocampalT')#left
TRvektor = ('RentorhinalT','RinferiortemporalT','RtransversetemporalT', 'RtemporalpoleT','RparahippocampalT','RfusiformT', 'RsuperiortemporalT', 'RmiddletemporalT')#right

#Frontal ROIS
Fvektor =('LfrontalpoleT', 'LparsorbitalisT','LparstriangularisT','LrostralmiddlefrontalT','LlateralorbitofrontalT','LmedialorbitofrontalT','LcaudalmiddlefrontalT',
'LcaudalanteriorcingulateT','LrostralanteriorcingulateT','LsuperiorfrontalT','LparsopercularisT')#left
FRvektor =('RfrontalpoleT', 'RparsorbitalisT','RparsopercularisT','RparstriangularisT', 'RsuperiorfrontalT','RmedialorbitofrontalT','RcaudalmiddlefrontalTN',
'RcaudalanteriorcingulateTN','RrostralanteriorcingulateTN','RlateralorbitofrontalT','RrostralmiddlefrontalT')#right

###################################################################################################################################################################################
###############CHANGE covriates SES_2/ without SES_2, RmeanT or LmeanT (right or left side) and Tvektor/TRvektor Fvektor/FRvektor in the loop and the plotRoi()###############
###################################################################################################################################################################################

predictor = 'Gruppenzugehörigkeit'
covariates = (' SES_2 + AlterJahreMonate  + RmeanT +  C(Standort,Treatment(reference="Dortmund")) + C(Geschlecht,Treatment(reference="männlich")) ') #change SES_2 and RmeanT or LmeanT + Probanden_ID

sns.set_style("darkgrid")

ols_df = pd.DataFrame()
for response in TRvektor: #change vektor
    res = smf.ols(f"{response} ~ {covariates} + {predictor}", data=zdf).fit() 
    res_df = format_ols_results(res)
    res_df['Areal'] = response
    ols_df = pd.concat([res_df,ols_df], axis=0, join='outer')
A = ols_df.loc['Gruppenzugehörigkeit']
plotRoi(FRvektor,predictor,'APARC_FREE','(Thickness)',A,5) #change vektor 
#multipletests(A['pvalues'],method='fdr_bh', is_sorted=False, returnsorted=False)   

########################################################
#####to have a look at the complete regression model####
########################################################

#problems with unequal variance or homoscedasticity one can use weighted least square  smf.wls or sm.gls (generalized least squares) or see below for robust regression 

#response = 'RtemporalpoleT' #'LinferiortemporalT'#'LparstriangularisT'# 'LfrontalpoleT'# # 'LinferiortemporalT' #'RtemporalpoleT'
#results = smf.ols(f"{response} ~ {covariates} + {predictor}", data=zdf).fit()
#results.summary()
#results.model.data.orig_exog[:] # Designmatrix


### Assumption Check <a class="anchor" id="assumchth"></a>

In [ ]:
###################################################################################
###############CHANGE Tvektor/TRvektor Fvektor/FRvektor in the loop ###############
###################################################################################

for response in FRvektor:
    res = smf.ols(f"{response} ~ {covariates} + {predictor}", data=zdf).fit()
    print('\n\n' +"\033[1m"+"\033[1;37;40m" ,response,"\033[0m")
    #print(res.summary())
    cls = LinearRegDiagnostic(res)
    vif, fig, ax = cls()
    print(vif)

In [ ]:
#robust regression (broken homoscedasticity assumption or unequal error variances assumption)
#Example regions 
vektor =('RparsopercularisT', 'RparstriangularisT' , 'RrostralanteriorcingulateTN' , 'RrostralmiddlefrontalT', 'LentorhinalT' , 'LtransversetemporalT', 'LtemporalpoleT','RparahippocampalT', 'RsuperiortemporalT')

import statsmodels.api as sm
zdf1 = zdf
zdf1 = zdf1.replace(['männlich','weiblich','Gießen','Dortmund'],[1.0,0.0,0.0,1.0])
data =zdf1

for response in vektor:
    data.endog = zdf1[response]
    data.exog = zdf1[['SES_2'  , 'AlterJahreMonate'  , 'eTIV' , 'Standort' , 'Geschlecht', 'Gruppenzugehörigkeit']]
    data.exog = sm.add_constant(data.exog)
    huber_t =sm.RLM(data.endog, data.exog, M=sm.robust.norms.HuberT())
    hub_results = huber_t.fit()
    print('\n\n' +"\033[1m"+"\033[1;37;40m" ,response,"\033[0m",'\n\n')
    print(hub_results.params)
    print(
        hub_results.summary(
            yname="y", xname=["var_%d" % i for i in range(len(hub_results.params))]
        )
    )

## R - Hierachial model Results <a class="anchor" id="res"></a>



In [ ]:
#################################
####Change Kernel to R Kernel####
#################################

setwd("") #set path to data
if (!require("pacman")) install.packages("pacman")
 pacman::p_load(bayestestR, dplyr,fastDummies, ggplot2,ggpubr,BayesFactor,rstanarm,varBF,bayesplot, see, performance, lme4, lmerTest, nlme, reshape, car, lattice, lsr, rcompanion)

dfR <- read.table("Aufbereitet_fam.csv",sep = ",",fileEncoding="UTF-8", header=T,dec = "." )
dfR <- dfR %>% mutate_at(c('RinsulaVN', 'LinsulaVN'), as.numeric)
zdfR <- dfR %>% mutate(across(where(is.double), scale))

zdfR <- dummy_cols(zdfR, select_columns = c('Gruppenzugehörigkeit', 'Standort','Geschlecht'))
zdfRwithout <- subset(zdfR,Geschwisterkind=="nein")

### Sample Comparsion <a class="anchor" id="samplecom"></a>

In [ ]:
#normal distributed -  Boxplot, QQPlot,histo with fitted curve, density distribution compare the observed distribution to what we would expect 
#Kind1_CBCL.Tsum has missing values 
defaultW <- getOption("warn") 
options(warn = -1) 

    for(i in c("AlterJahreMonate", "SES_2","Kind1_CBCL.Tsum")) {
        a <- ggplot(dfR, aes(x = Gruppenzugehörigkeit, y = .data[[i]])) +
            geom_boxplot(fill="lightgray") + 
            geom_dotplot(binaxis = "y", stackdir = "center", dotsize = 0.2,binwidth = 0.5)

        b <- ggqqplot(dfR, x = i,
                color = "Gruppenzugehörigkeit", 
                palette = c("#0073C2FF", "#FC4E07"),
                ggtheme = theme_pubclean())+
                ylab(i)+
                xlab("Theoretical Quantiles")
             

        c <- gghistogram(
                        dfR, x = i, y = "..density..",
                        add = "mean", rug = TRUE, bins=30,
                        fill = "Gruppenzugehörigkeit", palette = c("#00AFBB", "#E7B800"),
                        add_density = TRUE)
        d <- ggdensity(dfR, x = i, fill = "lightgray", title = i) +
                        coord_cartesian() +
                        stat_overlay_normal_density(color = "red", linetype = "dashed")     

        options(repr.plot.width=15, repr.plot.height=10)
        g <-     (ggarrange(a,b,c,d, 
            labels = c("A", "B","C","D"),
            ncol = 2, nrow = 2))
        print(annotate_figure(g, top = text_grob(paste0(i), 
               color = "red", face = "bold", size = 14)))
    }

options(warn = defaultW)  

In [ ]:
#Ttest on soziodata between groups
result <- list()
cohen <- list()
    for(i in c("AlterJahreMonate", "SES_2","Kind1_CBCL.Tsum")) {
        result[[i]] <- (t.test(dfR[[i]] ~ dfR$Gruppenzugehörigkeit, var.equal = FALSE, alternative = "two.sided"))
        cohen[[i]] <- cohensD(dfR[[i]] ~ dfR$Gruppenzugehörigkeit)
        print(i);print(result[[i]]);print(cohen[[i]]);cat("\n\n")
    }

In [ ]:
#Chi² or Fisher(better) for eduaction between groups
kreuztabelle <- xtabs (~ zdfR$Gruppenzugehörigkeit + zdfR$Kind1_schule)
n <- sum(kreuztabelle)
erwartete_häufigkeiten <- outer (rowSums(kreuztabelle), colSums(kreuztabelle)) / n
erwartete_häufigkeiten
#chisq.test(zdfR$Gruppenzugehörigkeit, zdfR$Kind1_schule)
fisher.test(zdfR$Gruppenzugehörigkeit, zdfR$Kind1_schule)


kreuztabelle <- xtabs (~ zdfR$Gruppenzugehörigkeit + zdfR$Geschlecht)
n <- sum(kreuztabelle)
erwartete_häufigkeiten <- outer (rowSums(kreuztabelle), colSums(kreuztabelle)) / n
erwartete_häufigkeiten
chisq.test(zdfR$Gruppenzugehörigkeit, zdfR$Geschlecht)
cohenW(zdfR$Gruppenzugehörigkeit, zdfR$Geschlecht)

### Hierarchical Volume model <a class="anchor" id="hiervol"></a>

In [ ]:
###############################################
#####InterceptOnly vs. RandomInterceptOnly##### 
########### Model Comparison ##################
###############################################

options("width"=200)
vektor = c("RightAmygdala","LeftAmygdala","RightHippocampus","LeftHippocampus","CCPosterior","CCMidPosterior","CCCentral","CCMidAnterior","CCAnterior","LinsulaVN","RinsulaVN")
interceptOnly=list()
randomInterceptOnly=list()
for(i in vektor) {
    set.seed(123)
    interceptOnly[[i]] <- gls(as.formula(paste0(i, "~" ,1 )), data=zdfR, method="ML")
    randomInterceptOnly[[i]] <- lme(as.formula(paste0(i, "~",  1)), random = ~1|Probanden_ID, data = zdfR, method="ML")
    print(i);
    print(anova(interceptOnly[[i]], randomInterceptOnly[[i]]))
    cat("\n\n\n");
}
#the random model is often not better

In [ ]:
######################################
#####Fullmodel vs. RandomFullmodel####
########### Model Comparison #########
######################################

options("width"=200)
vektor = c("RightAmygdala","LeftAmygdala","RightHippocampus","LeftHippocampus","CCPosterior","CCMidPosterior","CCCentral","CCMidAnterior","CCAnterior","LinsulaVN","RinsulaVN")
Fullmodel=list()
FullmodelrandomIntercept=list()
for(i in vektor) {
    set.seed(123)
    Fullmodel[[i]] <- gls(as.formula(paste0(i, " ~ SES_2 + AlterJahreMonate  + eTIV + Standort_Dortmund + Geschlecht_männlich  + Gruppenzugehörigkeit_EG" )), data=zdfR, method="ML")
    FullmodelrandomIntercept[[i]] <- lme(as.formula(paste0(i," ~  SES_2 + AlterJahreMonate  + eTIV + Standort_Dortmund + Geschlecht_männlich  + Gruppenzugehörigkeit_EG")), random = ~1|Probanden_ID, data = zdfR, method="ML")
    print(i);
    print(anova(Fullmodel[[i]], FullmodelrandomIntercept[[i]]))
    cat("\n\n\n");
}
#the random model is often not better

In [ ]:
####################################
#### RandomFullmodel Evaluation ####
####################################

options("width"=200)
vektor = c("RightAmygdala","LeftAmygdala","RightHippocampus","LeftHippocampus","CCPosterior","CCMidPosterior","CCCentral","CCMidAnterior","CCAnterior","LinsulaVN","RinsulaVN")
model=list()
for(i in vektor) {
    set.seed(123)
    #change SES_2
    model[[i]] <- lme(as.formula(paste0(i," ~ SES_2 + AlterJahreMonate  + eTIV + Standort_Dortmund + Geschlecht_männlich  + Gruppenzugehörigkeit_EG")), random = ~1|Probanden_ID, data = zdfR, method="ML")
    cat("####",i, "####\n");
    print(summary(model[[i]]));
    cat("\n");
    #print(intervals(model[[i]], 0.95))
    var_components <- VarCorr(model[[i]])
    cat("ICC:", as.numeric(var_components[1]) / (as.numeric(var_components[1]) + as.numeric(var_components[2])), "\n")
    print(performance(model[[i]]))
    cat("\n\n\n");
}
#the more complex model is used because it is more accurate regarding the data structure
#the simpler multiple regression model would be good enough (in the end it doesn´t change the tendencies)

### Hierarchical Volume model Assumption <a class="anchor" id="assumvol"></a>

In [ ]:
####################################
#### RandomFullmodel Assumption ####
####################################


#change used ROI vektor to the same as above

for(i in vektor) { #change vektor
  resid = resid(model[[i]])
  residn = resid(model[[i]], type ="normalized")
  fit = fitted(model[[i]])
  sresid = resid/sd(resid)
  sqrt_abs_resid <- sqrt(abs(resid))
  random_effects <- ranef(model[[i]])
  random_effects_df <- as.data.frame(random_effects)
  random_effects_df$Group <- rownames(random_effects_df)

  acf_values <- acf(resid, plot = FALSE)
# Convert ACF output to a data frame
  acf_df <- data.frame(
    Lag = acf_values$lag[, 1, 1],  # Lag values
    ACF = acf_values$acf[, 1, 1]  # Autocorrelation values
    )

options(repr.plot.width=22, repr.plot.height=18)


# Q-Q Plot for random intercepts
a <- ggplot(random_effects_df, aes(sample = random_effects_df[["(Intercept)"]])) +
  stat_qq() +
  stat_qq_line(color = "red") +
  labs(title = "Q-Q Plot of Random Intercepts", x = "Theoretical Quantiles", y = "Sample Quantiles")

b <- dotplot(random_effects_df[["(Intercept)"]] ~ random_effects_df$Group,
        main = "Random Intercept Distribution",
        xlab = "Groups (Families)", ylab = "Random Intercepts",
        scales = list(x = list(rot = 90)))  # Rotate x-axis labels for readability


# Identify potential outliers (absolute residual > 3)
outliers <- abs(residn) > 3
zdfR$outlier <- outliers  # Add a flag to the dataset

# Plot residuals to visualize
c <- ggplot(data = zdfR, aes(x = seq_along(residn), y = residn)) +
  geom_point(aes(color = outlier), alpha = 0.6) +
  geom_hline(yintercept = c(-3, 3), linetype = "dashed", color = "red") +
  labs(
    title = "Standardized Residuals with Outliers",
    x = "Observation Index",
    y = "Standardized Residuals"
  ) 

#non-linearity, unequal error variances, and outliers (e.g. residual vs fitted plot)
d <- ggplot(data=NULL,mapping=aes(x=fit,y=resid)) +
              geom_point(shape=1) +
              geom_hline(yintercept=0,linetype="dashed") +
              geom_smooth(color="red",linetype = "dashed",linewidth=0.5)+
              ylab("Residuals")+
              xlab("Fitted valus")+
              ggtitle("Residual vs. Fitted")

#qqPlot of residuals (should be normal distributed)
e <- ggplot(data=NULL, aes(sample = sresid)) +
              ggtitle("Normal Q-Q")+  
              ylab("Standardized residuals")+
              xlab("Theoretical Quantiles")+
              geom_qq(shape=1) +
              geom_qq_line(linetype = "dashed",color="red")
#autocorrelations of the residuals
f <- ggplot(acf_df, aes(x = Lag, y = ACF)) +
  geom_bar(stat = "identity", fill = "skyblue", color = "black") +
  geom_hline(yintercept = 0, linetype = "dashed") +
  geom_hline(yintercept = c(-1.96 / sqrt(length(resid)), 1.96 / sqrt(length(resid))), 
             linetype = "dotted", color = "red") +
  labs(title = "Autocorrelation of Residuals",
       x = "Lag",
       y = "Autocorrelation")

#Scale-Location
g <- ggplot(data = NULL, aes(x = fit, y = sqrt_abs_resid)) +
  geom_point(shape=1) +
  geom_smooth(method = "loess", se = FALSE, color = "red", linetype = "dashed") +
  labs(
    title = "Scale-Location Plot",
    x = "Fitted Values",
    y = "Sqrt(|Residuals|)"
  ) 

h <- (ggarrange(a,b,c,d,e,f,g, 
                  labels = c("A", "B","C","D","E","F","G"),
                  ncol = 2, nrow = 4))


print(annotate_figure(h, top = text_grob(paste0(i), 
                        color = "red", face = "bold", size = 14)))

}   

### Hierarchical Thickness model <a class="anchor" id="hierthick"></a>

In [ ]:
###############################################
#####InterceptOnly vs. RandomInterceptOnly##### 
##### Thickness Model Comparison ##############
###############################################

#Temporal ROIS
Tvektor = c('LentorhinalT','LfusiformT', 'LinferiortemporalT','LtransversetemporalT', 'LtemporalpoleT','LsuperiortemporalT','LmiddletemporalT','LparahippocampalT')#left
TRvektor = c('RentorhinalT','RinferiortemporalT','RtransversetemporalT', 'RtemporalpoleT','RparahippocampalT','RfusiformT', 'RsuperiortemporalT', 'RmiddletemporalT')#right

#Frontal ROIS
Fvektor =c('LfrontalpoleT', 'LparsorbitalisT','LparstriangularisT','LrostralmiddlefrontalT','LlateralorbitofrontalT','LmedialorbitofrontalT','LcaudalmiddlefrontalT',
'LcaudalanteriorcingulateT','LrostralanteriorcingulateT','LsuperiorfrontalT','LparsopercularisT')#left
FRvektor =c('RfrontalpoleT', 'RparsorbitalisT','RparsopercularisT','RparstriangularisT', 'RsuperiorfrontalT','RmedialorbitofrontalT','RcaudalmiddlefrontalTN',
'RcaudalanteriorcingulateTN','RrostralanteriorcingulateTN','RlateralorbitofrontalT','RrostralmiddlefrontalT')#right

interceptOnly=list()
randomInterceptOnly=list()
#####change vektor
for(i in Tvektor) { 
    set.seed(123)
    interceptOnly[[i]] <- gls(as.formula(paste0(i, "~" ,1 )), data=zdfR, method="ML")
    randomInterceptOnly[[i]] <- lme(as.formula(paste0(i, "~",  1)), random = ~1|Probanden_ID, data = zdfR, method="ML")
    print(i);
    print(anova(interceptOnly[[i]], randomInterceptOnly[[i]]))
    cat("\n\n\n");
}
#same as above

In [ ]:
######################################
#####Fullmodel vs. RandomFullmodel####
###### Thickness Model Comparison ####
######################################

#Temporal ROIS
Tvektor = c('LentorhinalT','LfusiformT', 'LinferiortemporalT','LtransversetemporalT', 'LtemporalpoleT','LsuperiortemporalT','LmiddletemporalT','LparahippocampalT')#left
TRvektor = c('RentorhinalT','RinferiortemporalT','RtransversetemporalT', 'RtemporalpoleT','RparahippocampalT','RfusiformT', 'RsuperiortemporalT', 'RmiddletemporalT')#right

#Frontal ROIS
Fvektor =c('LfrontalpoleT', 'LparsorbitalisT','LparstriangularisT','LrostralmiddlefrontalT','LlateralorbitofrontalT','LmedialorbitofrontalT','LcaudalmiddlefrontalT',
'LcaudalanteriorcingulateT','LrostralanteriorcingulateT','LsuperiorfrontalT','LparsopercularisT')#left
FRvektor =c('RfrontalpoleT', 'RparsorbitalisT','RparsopercularisT','RparstriangularisT', 'RsuperiorfrontalT','RmedialorbitofrontalT','RcaudalmiddlefrontalTN',
'RcaudalanteriorcingulateTN','RrostralanteriorcingulateTN','RlateralorbitofrontalT','RrostralmiddlefrontalT')#right

Fullmodel=list()
FullmodelrandomIntercept=list()
#####change vektor
for(i in Tvektor) {  
    set.seed(123)
    #change SES_2 and RmeanT or LmeanT
    Fullmodel[[i]] <- gls(as.formula(paste0(i, " ~ SES_2 + AlterJahreMonate  + LmeanT + Standort_Dortmund + Geschlecht_männlich  + Gruppenzugehörigkeit_EG" )), data=zdfR, method="ML")
    FullmodelrandomIntercept[[i]] <- lme(as.formula(paste0(i," ~  SES_2 + AlterJahreMonate  + LmeanT + Standort_Dortmund + Geschlecht_männlich  + Gruppenzugehörigkeit_EG")), random = ~1|Probanden_ID, data = zdfR, method="ML")
    print(i);
    print(anova(Fullmodel[[i]], FullmodelrandomIntercept[[i]]))
    cat("\n\n\n");
}

In [ ]:
##############################################
#### RandomFullmodel Evaluation Thickness ####
##############################################

#Temporal ROIS
Tvektor = c('LentorhinalT','LfusiformT', 'LinferiortemporalT','LtransversetemporalT', 'LtemporalpoleT','LsuperiortemporalT','LmiddletemporalT','LparahippocampalT')#left
TRvektor = c('RentorhinalT','RinferiortemporalT','RtransversetemporalT', 'RtemporalpoleT','RparahippocampalT','RfusiformT', 'RsuperiortemporalT', 'RmiddletemporalT')#right

#Frontal ROIS
Fvektor =c('LfrontalpoleT', 'LparsorbitalisT','LparstriangularisT','LrostralmiddlefrontalT','LlateralorbitofrontalT','LmedialorbitofrontalT','LcaudalmiddlefrontalT',
'LcaudalanteriorcingulateT','LrostralanteriorcingulateT','LsuperiorfrontalT','LparsopercularisT')#left
FRvektor =c('RfrontalpoleT', 'RparsorbitalisT','RparsopercularisT','RparstriangularisT', 'RsuperiorfrontalT','RmedialorbitofrontalT','RcaudalmiddlefrontalTN',
'RcaudalanteriorcingulateTN','RrostralanteriorcingulateTN','RlateralorbitofrontalT','RrostralmiddlefrontalT')#right

model=list()

#####change vektor
for(i in Fvektor) {
    set.seed(123)
    #change SES_2 and RmeanT or LmeanT
    model[[i]] <- lme(as.formula(paste0(i," ~ SES_2 +  AlterJahreMonate  + LmeanT + Standort_Dortmund + Geschlecht_männlich  + Gruppenzugehörigkeit_EG")), random = ~1|Probanden_ID, data = zdfR, method="ML")
    cat("####",i, "####\n");
    print(summary(model[[i]]));
    cat("\n");
    #print(intervals(model[[i]], 0.95))
    var_components <- VarCorr(model[[i]])
    cat("ICC:", as.numeric(var_components[1]) / (as.numeric(var_components[1]) + as.numeric(var_components[2])))
    cat("\n\n\n");
}

### Hierarchical Thickness model Assumption <a class="anchor" id="assumthick"></a>

In [ ]:
####################################
#### RandomFullmodel Assumption ####
####################################

#### change used ROI vektor to the same as above ####
for(i in FRvektor) { 
  resid = resid(model[[i]])
  residn = resid(model[[i]], type ="normalized")
  fit = fitted(model[[i]])
  sresid = resid/sd(resid)
  sqrt_abs_resid <- sqrt(abs(resid))
  random_effects <- ranef(model[[i]])
  random_effects_df <- as.data.frame(random_effects)
  random_effects_df$Group <- rownames(random_effects_df)

  acf_values <- acf(resid, plot = FALSE)
# Convert ACF output to a data frame
  acf_df <- data.frame(
    Lag = acf_values$lag[, 1, 1],  # Lag values
    ACF = acf_values$acf[, 1, 1]  # Autocorrelation values
    )

options(repr.plot.width=22, repr.plot.height=18)

# Q-Q Plot for random intercepts
a <- ggplot(random_effects_df, aes(sample = random_effects_df[["(Intercept)"]])) +
  stat_qq() +
  stat_qq_line(color = "red") +
  labs(title = "Q-Q Plot of Random Intercepts", x = "Theoretical Quantiles", y = "Sample Quantiles")

b <- dotplot(random_effects_df[["(Intercept)"]] ~ random_effects_df$Group,
        main = "Random Intercept Distribution",
        xlab = "Groups (Families)", ylab = "Random Intercepts",
        scales = list(x = list(rot = 90)))  # Rotate x-axis labels for readability


# Identify potential outliers (absolute residual > 3)
outliers <- abs(residn) > 3
zdfR$outlier <- outliers  # Add a flag to the dataset

# Plot residuals to visualize
c <- ggplot(data = zdfR, aes(x = seq_along(residn), y = residn)) +
  geom_point(aes(color = outlier), alpha = 0.6) +
  geom_hline(yintercept = c(-3, 3), linetype = "dashed", color = "red") +
  labs(
    title = "Standardized Residuals with Outliers",
    x = "Observation Index",
    y = "Standardized Residuals"
  ) 

#non-linearity, unequal error variances, and outliers (e.g. residual vs fitted plot)
d <- ggplot(data=NULL,mapping=aes(x=fit,y=resid)) +
              geom_point(shape=1) +
              geom_hline(yintercept=0,linetype="dashed") +
              geom_smooth(color="red",linetype = "dashed",linewidth=0.5)+
              ylab("Residuals")+
              xlab("Fitted valus")+
              ggtitle("Residual vs. Fitted")

#qqPlot of residuals (should be normal distributed)
e <- ggplot(data=NULL, aes(sample = sresid)) +
              ggtitle("Normal Q-Q")+  
              ylab("Standardized residuals")+
              xlab("Theoretical Quantiles")+
              geom_qq(shape=1) +
              geom_qq_line(linetype = "dashed",color="red")
#autocorrelations of the residuals
f <- ggplot(acf_df, aes(x = Lag, y = ACF)) +
  geom_bar(stat = "identity", fill = "skyblue", color = "black") +
  geom_hline(yintercept = 0, linetype = "dashed") +
  geom_hline(yintercept = c(-1.96 / sqrt(length(resid)), 1.96 / sqrt(length(resid))), 
             linetype = "dotted", color = "red") +
  labs(title = "Autocorrelation of Residuals",
       x = "Lag",
       y = "Autocorrelation")

#Scale-Location
g <- ggplot(data = NULL, aes(x = fit, y = sqrt_abs_resid)) +
  geom_point(shape=1) +
  geom_smooth(method = "loess", se = FALSE, color = "red", linetype = "dashed") +
  labs(
    title = "Scale-Location Plot",
    x = "Fitted Values",
    y = "Sqrt(|Residuals|)"
  ) 

h <- (ggarrange(a,b,c,d,e,f,g, 
                  labels = c("A", "B","C","D","E","F","G"),
                  ncol = 2, nrow = 4))


print(annotate_figure(h, top = text_grob(paste0(i), 
                        color = "red", face = "bold", size = 14)))

}

### Bayesian Hierarchical model <a class="anchor" id="bayesueber"></a>


#### Bayesian Hierarchical Volume model <a class="anchor" id="bayesvolu"></a>

In [ ]:
### Compare the Fullmodel against the RandomFullmodel ###
#the following computations can take long (up to 20-30 min)

locationpa = 0 # priors (0,1/sqrt(2)); (0,1); (0,sqrt(2)); (0,10)
scalepa = 1

vektor = c("RightAmygdala","LeftAmygdala","RightHippocampus","LeftHippocampus","CCPosterior","CCMidPosterior","CCCentral","CCMidAnterior","CCAnterior","LinsulaVN","RinsulaVN")
Fullmodel=list()
Simplemodel=list()
for(i in vektor) {
    set.seed(123)
    #change SES_2
    Fullmodel[[i]] <- stan_glmer(paste0(i, " ~ SES_2 + AlterJahreMonate  + eTIV + Standort_Dortmund + Geschlecht_männlich  + Gruppenzugehörigkeit_EG + (1 | Probanden_ID)" )  , data = zdfR,  
        prior=normal(locationpa,scalepa,autoscale=TRUE), 
        prior_intercept=normal(locationpa,scalepa,autoscale=TRUE),  
        refresh = 0,
        iter = 15000, #15000 iterations-5000warmups*5chains = 50000
        warmup =5000,
        chains =5,
        QR = TRUE,
        adapt_delta=0.99)
    Simplemodel[[i]] <- stan_glm(paste0(i, " ~ SES_2 + AlterJahreMonate  + eTIV + Standort_Dortmund + Geschlecht_männlich  + Gruppenzugehörigkeit_EG" )  , data = zdfR,  
        prior=normal(locationpa,scalepa,autoscale=TRUE), 
        prior_intercept=normal(locationpa,scalepa,autoscale=TRUE),  
        refresh = 0,
        iter = 15000, #15000 iterations-5000warmups*5chains = 50000
        warmup =5000,
        chains =5,
        QR = TRUE,
        adapt_delta=0.99)
    
    print(i);cat("\n");
    #print(tidyMCMC(model[[i]], conf.int = TRUE, conf.method = "HPDinterval"))
    #print(check_collinearity(model[[i]]))
    #print(describe_posterior(model[[i]], priors=TRUE, centrality="all", ci_method="hdi",test=c("p_direction", "rope","bf","pd","ps"), diagnostic="all",ci=0.95,rope_ci=1,rope_range = "default"))
    #print(prior_summary(model[[i]]))
    #print(bf_rope(model[[i]])) #BFrope
    #print(point_estimate(model[[i]]))
    print(loo_compare(loo(Fullmodel[[i]],k_threshold = 0.7), loo(Simplemodel[[i]],k_threshold = 0.7))) 
    cat("\n\n")
    #elpd_diff > 2 * se_diff -> difference s large enough that´s unlike to be due random noise
    #it´s to times because it approx to the 95%CI
}
#same as above the complex model seems not rly more usefull but it is more acurate regarding the data structure

In [ ]:
### RandomFullmodel ###
locationpa = 0 # priors (0,1/sqrt(2)); (0,1); (0,sqrt(2)); (0,10)
scalepa = 1

vektor = c("RightAmygdala","LeftAmygdala","RightHippocampus","LeftHippocampus","CCPosterior","CCMidPosterior","CCCentral","CCMidAnterior","CCAnterior","LinsulaVN","RinsulaVN")
model=list()
for(i in vektor) {
    set.seed(123)
    #change SES_2
    model[[i]] <- stan_glmer(paste0(i, " ~ SES_2 + AlterJahreMonate  + eTIV + Standort_Dortmund + Geschlecht_männlich  + Gruppenzugehörigkeit_EG + (1 | Probanden_ID)" )  , data = zdfR,  
        prior=normal(locationpa,scalepa,autoscale=TRUE), 
        prior_intercept=normal(locationpa,scalepa,autoscale=TRUE),  
        refresh = 0,
        iter = 15000, #15000 iterations-5000warmups*5chains = 50000
        warmup =5000,
        chains =5,
        QR = TRUE,
        adapt_delta=0.999)
    
    print(i);cat("\n");
    #print(tidyMCMC(model[[i]], conf.int = TRUE, conf.method = "HPDinterval"))
    #print(check_collinearity(model[[i]]))
    print(describe_posterior(model[[i]], priors=TRUE, centrality="all", ci_method="hdi",test=c("p_direction", "rope","bf","pd","ps"), diagnostic="all",ci=0.95,rope_ci=1,rope_range = "default"))
    #print(prior_summary(model[[i]]))
    #print(bf_rope(model[[i]])) #BFrope
    #print(point_estimate(model[[i]]))
    print(performance(model[[i]]))
    cat("\n\n")
}

#### Posterior Volume <a class="anchor" id="posteriorv"></a>

In [ ]:
#check visually posteriors

for(ROI in vektor) {
  posterior <- as.matrix(model[[ROI]])
  plot_title <- ggtitle("Posterior distributions",
                        "with medians and 95% intervals")

 #############change "SES_2"
  a <- mcmc_areas(posterior,
            pars = c( "SES_2","AlterJahreMonate", "Standort_Dortmund", "Geschlecht_männlich","Gruppenzugehörigkeit_EG","eTIV"), #change SES_2
            prob = 0.95) + plot_title

  b <- plot(bayesfactor_parameters(model[[ROI]],parameter="Gruppenzugehörigkeit_EG")) +
    scale_color_material() +
    scale_fill_material() +
    ggtitle("Posterior and Prior for Group")   #results in a plot presenting the prior and posterior distributions for parameter. When a point null was tested, two dots represent the density of the null at the value - the ratio of their heights is the value of the Savage-Dickey Bayes factor

    # Extract posterior samples for random effects
  random_effects <- ranef(model[[ROI]], condVar = TRUE)
  random_effects_df <- as.data.frame(random_effects)
  random_effects_df$Lower <- random_effects_df$condval - 1.96 * random_effects_df$condsd  # 95% CI lower
  random_effects_df$Upper <- random_effects_df$condval + 1.96 * random_effects_df$condsd  # 95% CI upper

  c <- ggplot(random_effects_df, aes(x = condval, y = reorder(grp, condval))) +
        geom_point() +
        geom_errorbarh(aes(xmin = Lower, xmax = Upper), height = 0.2) +
        labs(title = "Caterpillar Plot of Random Effects",
            x = "Random Effect Estimate (Posterior Mean)",
            y = "Probanden_ID") 

  options(repr.plot.width=19, repr.plot.height=8)
  g <- ggarrange(a,b,c, ncol=3,nrow=1)
  print(annotate_figure(g, top = text_grob(paste0(ROI), 
          color = "red", face = "bold", size = 14)))
}

#### Assumption and Model Check Volume <a class="anchor" id="assum"></a>

In [ ]:
###change SES_2
Predictor = c("SES_2", "AlterJahreMonate", "Standort_Dortmund", "Geschlecht_männlich","Gruppenzugehörigkeit_EG","eTIV")
for(i in vektor) {
  resid = resid(model[[i]])
  fit = fitted(model[[i]])
  sresid = resid/sd(resid)

  options(repr.plot.width=22, repr.plot.height=18)

  #non-linearity, unequal error variances, and outliers (e.g. residual vs fitted plot)
  a <- ggplot(data=NULL,mapping=aes(x=fit,y=resid)) +
              geom_point(shape=1) +
              geom_hline(yintercept=0,linetype="dashed") +
              geom_smooth(color="red",linetype = "dashed",linewidth=0.5)+
              ylab("Residuals")+
              xlab("Fitted valus")+
              ggtitle("Residual vs. Fitted")

  #qqPlot of residuals (should be normal distributed)
  b <- ggplot(data=NULL, aes(sample = sresid)) +
              ggtitle("Normal Q-Q")+  
              ylab("Standardized residuals")+
              xlab("Theoretical Quantiles")+
              geom_qq(shape=1) +
              geom_qq_line(linetype = "dashed",color="red")

  #Posterior predictive check (makes model sense to explain data)
  c <- pp_check(model[[i]], nreps=100) + xlab(paste0(i)) + ggtitle("Posterior predictive check") + theme(plot.title = element_text(hjust = 0.06))

  #this compares the posterior estimate for each parameter against the associated prior. 
  #If the spread of the priors is small relative to the posterior, then it is likely that the priors are too influential.
  d <- posterior_vs_prior(model[[i]], color_by = "vs", group_by = TRUE, pars = c(Predictor), 
                          facet_args = list(scales = "free_y")) + ggtitle("Posterior vs. Prior")

  color_scheme_set("mix-blue-red")

  #take a look at the posteriors for each chain and the trace 
  #Trace plots show no evidence that the chains have not reasonably traversed the entire multidimensional parameter space
  e <- mcmc_combo(model[[i]],
                  combo = c("dens_overlay", "trace"),
                  pars = c(Predictor), 
                  gg_theme = ggplot2::theme_gray()) 

  #autocorrelation check 
  f <- mcmc_acf(model[[i]], pars = c(Predictor))
 
  g <- (ggarrange(a,b,c,d,e,f, 
                  labels = c("A", "B","C","D","E","F"),
                  ncol = 2, nrow = 3))

  print(annotate_figure(g, top = text_grob(paste0(i), 
                        color = "red", face = "bold", size = 14)))
}


### Bayesian Hierarchical Thickness model <a class="anchor" id="bayesthickness"></a>



In [ ]:
####################################################################################################
###check mean thickness for right and left hemisphere --> RmeanT or LmeanT and with/without SES_2### 
####################################################################################################
    #z-transformed data = zdfR , without Siblings = zdfRwithout, dfR raw untransformed data 
    #change data in the model if you interested in with or without siblings

#Temporal ROIS
Tvektor = c('LentorhinalT','LfusiformT', 'LinferiortemporalT','LtransversetemporalT', 'LtemporalpoleT','LsuperiortemporalT','LmiddletemporalT','LparahippocampalT')#left
TRvektor = c('RentorhinalT','RinferiortemporalT','RtransversetemporalT', 'RtemporalpoleT','RparahippocampalT','RfusiformT', 'RsuperiortemporalT', 'RmiddletemporalT')#right
#Frontal ROIS
Fvektor =c('LfrontalpoleT', 'LparsorbitalisT','LparstriangularisT','LrostralmiddlefrontalT','LlateralorbitofrontalT','LmedialorbitofrontalT','LcaudalmiddlefrontalT',
'LcaudalanteriorcingulateT','LrostralanteriorcingulateT','LsuperiorfrontalT','LparsopercularisT')#left
FRvektor =c('RfrontalpoleT', 'RparsorbitalisT','RparsopercularisT','RparstriangularisT', 'RsuperiorfrontalT','RmedialorbitofrontalT','RcaudalmiddlefrontalTN',
'RcaudalanteriorcingulateTN','RrostralanteriorcingulateTN','RlateralorbitofrontalT','RrostralmiddlefrontalT')#right

#for those ROIs change chains = 8 or higher and take some more iterations like 30000 because BFMI is low suggesting difficulties efficiently exploring the posterior distribution
#could be because correlations between parameters or the model is complex (we already found out that the simpler model had better fit indices but this model is more "acurate" regarding the data
#But ESS and Rhat good enough so that the model has likely converged and is sampling effectivley so therefore no critical problem 
#the Tendencies are the same using a non hierarchical model and the model check looks a lot better

diffic = c('LtransversetemporalT','LrostralanteriorcingulateT','LparstriangularisT','LparsorbitalisT','LtransversetemporalT')
diffic1 = c('RmiddletemporalT') 

#priors (0,1/sqrt(2)); (0,1); (0,sqrt(2)); (0,10)
locationpa = 0 
scalepa = 1

model=list()
#change vektor
for(i in Fvektor) {
    set.seed(123)
    #change SES_2 and mean Thickness right or left: LmeanT/RmeanT
    model[[i]] <- stan_glmer(paste0(i, " ~ SES_2 + AlterJahreMonate  + LmeanT + Standort_Dortmund + Geschlecht_männlich + Gruppenzugehörigkeit_EG + (1 | Probanden_ID)" )  , data = zdfR,  
        prior=normal(locationpa,scalepa,autoscale=TRUE), 
        prior_intercept=normal(locationpa,scalepa,autoscale=TRUE),  
        refresh = 0,
        iter = 15000, #15000 iterations-5000warmups*5chains = 50000
        warmup =5000,
        chains =5,
        QR = TRUE,
        adapt_delta=0.999)
    
    print(i);cat("\n");
    #print(tidyMCMC(model[[i]], conf.int = TRUE, conf.method = "HPDinterval"))
    #print(check_collinearity(model[[i]]))
    print(describe_posterior(model[[i]], priors=TRUE, centrality="all", ci_method="hdi",test=c("p_direction", "rope","bf","pd","ps"), diagnostic="all",ci=0.95,rope_ci=1,rope_range = "default"))
    #print(prior_summary(model[[i]]))
    #print(bf_rope(model[[i]])) #BFrope
    #print(point_estimate(model[[i]]))
    print(performance(model[[i]]))
    cat("\n\n")
}

### Posterior Thickness <a class="anchor" id="posteriort"></a>

In [ ]:
###check visually Posteriors###

#change used ROI vektor to the same as above
for(ROI in TRvektor) {
  posterior <- as.matrix(model[[ROI]])
  plot_title <- ggtitle("Posterior distributions",
                        "with medians and 95% intervals")
  a <- mcmc_areas(posterior,
            #change covariates (SES_2/without_SES_2 and RmeanT or LmeanT like above)
            pars = c( "SES_2","AlterJahreMonate", "Standort_Dortmund", "Geschlecht_männlich","Gruppenzugehörigkeit_EG","RmeanT"),
            prob = 0.95) + plot_title

  b <- plot(bayesfactor_parameters(model[[ROI]],parameter="Gruppenzugehörigkeit_EG")) +
    scale_color_material() +
    scale_fill_material() +
    ggtitle("Posterior and Prior for Group")   #results in a plot presenting the prior and posterior distributions for parameter. When a point null was tested, two dots represent the density of the null at the value - the ratio of their heights is the value of the Savage-Dickey Bayes factor

  # Extract posterior samples for random effects
  random_effects <- ranef(model[[ROI]], condVar = TRUE)
  random_effects_df <- as.data.frame(random_effects)
  random_effects_df$Lower <- random_effects_df$condval - 1.96 * random_effects_df$condsd  # 95% CI lower
  random_effects_df$Upper <- random_effects_df$condval + 1.96 * random_effects_df$condsd  # 95% CI upper

  c <- ggplot(random_effects_df, aes(x = condval, y = reorder(grp, condval))) +
        geom_point() +
        geom_errorbarh(aes(xmin = Lower, xmax = Upper), height = 0.2) +
        labs(title = "Caterpillar Plot of Random Effects",
            x = "Random Effect Estimate (Posterior Mean)",
            y = "Probanden_ID") 

  options(repr.plot.width=19, repr.plot.height=8)
  g <- ggarrange(a,b,c, ncol=3,nrow=1)
  print(annotate_figure(g, top = text_grob(paste0(ROI), 
          color = "red", face = "bold", size = 14)))
}

#### Assumption and Model Check Thickness <a class="anchor" id="assumT"></a>

In [ ]:
###check model assumption###
#change used ROI vektor to the same as above

#change covariates (SES_2/without_SES_2 and RmeanT or LmeanT like above)
Predictor = c( "SES_2","AlterJahreMonate", "Standort_Dortmund", "Geschlecht_männlich","Gruppenzugehörigkeit_EG","RmeanT")
#change vektor
for(i in TRvektor) {
  resid = resid(model[[i]])
  fit = fitted(model[[i]])
  sresid = resid/sd(resid)

  options(repr.plot.width=22, repr.plot.height=18)

  #non-linearity, unequal error variances, and outliers (e.g. residual vs fitted plot)
  a <- ggplot(data=NULL,mapping=aes(x=fit,y=resid)) +
              geom_point(shape=1) +
              geom_hline(yintercept=0,linetype="dashed") +
              geom_smooth(color="red",linetype = "dashed",linewidth=0.5)+
              ylab("Residuals")+
              xlab("Fitted valus")+
              ggtitle("Residual vs. Fitted")

  #qqPlot of residuals (should be normal distributed)
  b <- ggplot(data=NULL, aes(sample = sresid)) +
              ggtitle("Normal Q-Q")+  
              ylab("Standardized residuals")+
              xlab("Theoretical Quantiles")+
              geom_qq(shape=1) +
              geom_qq_line(linetype = "dashed",color="red")

  #Posterior predictive check (makes model sense to explain data)
  c <- pp_check(model[[i]], nreps=100) + xlab(paste0(i)) + ggtitle("Posterior predictive check") + theme(plot.title = element_text(hjust = 0.06))

  #this compares the posterior estimate for each parameter against the associated prior. 
  #If the spread of the priors is small relative to the posterior, then it is likely that the priors are too influential.
  d <- posterior_vs_prior(model[[i]], color_by = "vs", group_by = TRUE, pars = c(Predictor), 
                          facet_args = list(scales = "free_y")) + ggtitle("Posterior vs. Prior")

  color_scheme_set("mix-blue-red")

  #take a look at the posteriors for each chain and the trace 
  #Trace plots show no evidence that the chains have not reasonably traversed the entire multidimensional parameter space
  e <- mcmc_combo(model[[i]],
                  combo = c("dens_overlay", "trace"),
                  pars = c(Predictor), 
                  gg_theme = ggplot2::theme_gray()) 

  #autocorrelation check 
  f <- mcmc_acf(model[[i]], pars = c(Predictor))
 
  g <- (ggarrange(a,b,c,d,e,f, 
                  labels = c("A", "B","C","D","E","F"),
                  ncol = 2, nrow = 3))

  print(annotate_figure(g, top = text_grob(paste0(i), 
                        color = "red", face = "bold", size = 14)))
}